Ingest customer source data into the RetailPulse Bronzengest customer source data into the RetailPulse Bronze

In [0]:
dbutils.widgets.text("environment","")


In [0]:
env_var = dbutils.widgets.get("environment").upper()
print(env_var)
allowed_env_vars = {'DEV','TEST','PROD'}

if env_var not in allowed_env_vars:
    raise ValueError(f"Invalid environment: '{env_var}' must be one of {allowed_env_vars}")


                      

In [0]:
config_table = f"retailpulse_config.{env_var.lower()}.databricks_configs"
print(config_table)
note_book_name_var = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split("/")[-1]

print(note_book_name_var)

config_df = spark.sql(f"""select * from {config_table}  where notebook_name= '{note_book_name_var}' """)
config_df.display()

In [0]:
config = {
    row["config_key"]:row["config_value"]

    for row in config_df.collect()
 
}
print(config)

In [0]:
source_path = config["source_path"]
target_path = config['target_path']
source_format = config["source_format"]
target_format = config["target_format"]
load_type = config["load_type"]
header = config["header"].strip().lower()
schema_enforcement =config['schema_enforcement']

print(f"source_path :  '{source_path}' ")
print(f"target_path :  '{target_path}' ")
print(f"source_format :  '{source_format}' ")
print(f"target_format :  '{target_format}' ")
print(f"load_type :  '{load_type}' ")
print(f"header :  '{header}' ")
print(f"schema_enforcement :  '{schema_enforcement}' ")



In [0]:
if not source_path :
    print("source path sould not be empty")

if not target_path:
    print("target path sould not be empty")

if not source_format:
    print("source format sould not be empty")

if not target_format:
    print("target format sould not be empty")

In [0]:
customer_df = ( spark.read.format(source_format).option("header",header)\
            .option("inferschema","false")\
            .load(source_path) 
)
customer_df.head(2)

In [0]:
customer_df.write.format(target_format)\
    .mode(load_type)\
    .save(target_path)

In [0]:
bronze_customer_df = spark.read.parquet(target_path)
print(bronze_customer_df.count() )
print(customer_df.count() )